In [1]:
import json, re, math, statistics, html, unicodedata
from pathlib import Path
import pandas as pd, numpy as np
from collections import Counter, defaultdict

path = Path('/workspace/rsi_titles_abstracts_2021-2026.jsonl')
rows=[]
with path.open(encoding='utf-8') as f:
    for i,line in enumerate(f,1):
        obj=json.loads(line)
        obj['_line']=i
        rows.append(obj)
df=pd.DataFrame(rows)
print("ANALYSIS PLAN")
print("1. Validate records and normalize markup/Unicode without changing words.")
print("2. Measure title word/character lengths and structural markers on all records, with sensitivity checks excluding non-research notices.")
print("3. Define a transparent abstract/title keyword heuristic for instrument-development papers; quantify title naming of instrument class, technique, and application.")
print("4. Rank close comparators using topic keywords and TF-IDF similarity across titles+abstracts, then manually inspect the highest-ranked records.")
print("5. Compare the proposed title to empirical distributions and corpus terminology; propose RSI-style alternatives.")
print("Limitations: classifications and noun-phrase/sentence structure are heuristic; the corpus ends in 2026 per the supplied Crossref export, and Crossref metadata may contain notices/non-articles.")
print("\nShape:", df.shape)
print("Columns:", df.columns.tolist())
print("Missing/empty titles:", int(df.title.fillna('').str.strip().eq('').sum()))
print("Missing/empty abstracts:", int(df.abstract.fillna('').str.strip().eq('').sum()))
print("Duplicate DOIs:", int(df.doi.duplicated().sum()))
print("Years:", df.year.value_counts().sort_index().to_dict())
print(df.head(3).to_string(index=False))

ANALYSIS PLAN
1. Validate records and normalize markup/Unicode without changing words.
2. Measure title word/character lengths and structural markers on all records, with sensitivity checks excluding non-research notices.
3. Define a transparent abstract/title keyword heuristic for instrument-development papers; quantify title naming of instrument class, technique, and application.
4. Rank close comparators using topic keywords and TF-IDF similarity across titles+abstracts, then manually inspect the highest-ranked records.
5. Compare the proposed title to empirical distributions and corpus terminology; propose RSI-style alternatives.
Limitations: classifications and noun-phrase/sentence structure are heuristic; the corpus ends in 2026 per the supplied Crossref export, and Crossref metadata may contain notices/non-articles.

Shape: (4408, 5)
Columns: ['doi', 'year', 'title', 'abstract', '_line']
Missing/empty titles: 0
Missing/empty abstracts: 84
Duplicate DOIs: 0
Years: {2021: 374, 202

In [2]:
# Normalize title text and compute length/structure metrics.
def clean_text(s):
    s=html.unescape(str(s or ''))
    s=re.sub(r'<[^>]+>', ' ', s)
    s=unicodedata.normalize('NFKC', s)
    return re.sub(r'\s+', ' ', s).strip()

def words(s):
    # Hyphenated/slashed technical compounds count as one orthographic token.
    return re.findall(r"[A-Za-z0-9]+(?:[–—\-/'][A-Za-z0-9]+)*", clean_text(s))

df['title_clean']=df.title.map(clean_text)
df['abstract_clean']=df.abstract.map(clean_text)
df['n_words']=df.title_clean.map(lambda s: len(words(s)))
df['n_chars']=df.title_clean.str.len()
# Metadata/non-research sensitivity filter, based on title labels.
notice_pat=r'^(erratum|corrigendum|publisher.s note|editorial|preface|review of scientific instruments new products|new products|retraction)\b'
df['notice']=df.title_clean.str.lower().str.contains(notice_pat, regex=True)
research=df[~df.notice].copy()

for label,x in [('All records',df),('Excluding labeled notices',research)]:
    print('\n',label,'n=',len(x))
    for col in ['n_words','n_chars']:
        q=x[col].quantile([.05,.25,.5,.75,.95]).round(1).to_dict()
        print(col, 'mean=',round(x[col].mean(),1),'SD=',round(x[col].std(),1),'q=',q,'min/max=',(x[col].min(),x[col].max()))

struct = pd.DataFrame(index=research.index)
struct['colon']=research.title_clean.str.contains(':')
struct['question']=research.title_clean.str.endswith('?')
struct['leading_gerund']=research.title_clean.str.match(r'^[A-Z][a-z]+ing\b')
struct['leading_article']=research.title_clean.str.match(r'^(A|An|The)\b')
struct['method_lead']=research.title_clean.str.match(r'^(A|An) (method|approach|technique|system|instrument|device|apparatus|setup|platform|framework|model|sensor|detector|source|spectrometer|microscope|diagnostic)\b', case=False)
print('\nStructure frequencies (notices excluded):')
for c in struct:
    print(c, int(struct[c].sum()), f"({100*struct[c].mean():.1f}%)")
print('\nLeading gerund examples:')
print(research.loc[struct.leading_gerund,['title_clean','doi']].head(30).to_string(index=False))

proposed="Retrofitting a commercial RF induction generator into a computer-controlled, vacuum-integrated annealing system for reactive-metal grain growth"
pw, pc=len(words(proposed)),len(proposed)
from scipy.stats import percentileofscore
print('\nProposed:',pw,'words,',pc,'characters')
print('Percentiles among research-like records: words',round(percentileofscore(research.n_words,pw,kind='mean'),1),
      'chars',round(percentileofscore(research.n_chars,pc,kind='mean'),1))
print('Counts >= proposed:',int((research.n_words>=pw).sum()),int((research.n_chars>=pc).sum()),'of',len(research))
print('Leading gerund exact Retrofitting:',int(research.title_clean.str.match(r'^Retrofitting\b',case=False).sum()))

/tmp/ipykernel_20/3610144648.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['notice']=df.title_clean.str.lower().str.contains(notice_pat, regex=True)



 All records n= 4408
n_words mean= 13.1 SD= 4.1 q= {0.05: 7.0, 0.25: 10.0, 0.5: 13.0, 0.75: 16.0, 0.95: 20.0} min/max= (np.int64(3), np.int64(44))
n_chars mean= 101.3 SD= 30.5 q= {0.05: 55.0, 0.25: 80.0, 0.5: 99.0, 0.75: 120.0, 0.95: 154.0} min/max= (np.int64(19), np.int64(279))

 Excluding labeled notices n= 4285
n_words mean= 13.1 SD= 4.0 q= {0.05: 7.0, 0.25: 10.0, 0.5: 13.0, 0.75: 16.0, 0.95: 20.0} min/max= (np.int64(3), np.int64(44))
n_chars mean= 101.4 SD= 29.4 q= {0.05: 58.0, 0.25: 80.0, 0.5: 99.0, 0.75: 119.0, 0.95: 152.0} min/max= (np.int64(19), np.int64(279))

Structure frequencies (notices excluded):
colon 256 (6.0%)
question 2 (0.0%)
leading_gerund 232 (5.4%)
leading_article 949 (22.1%)
method_lead 32 (0.7%)

Leading gerund examples:
                                                                                                                                             title_clean               doi
                                                                         


Proposed: 16 words, 143 characters
Percentiles among research-like records: words 78.2 chars 91.8
Counts >= proposed: 1072 362 of 4285
Leading gerund exact Retrofitting: 0


In [3]:
# Transparent heuristic subset: development/build papers whose abstracts explicitly report presenting/developing/designing/building an instrument-like object.
A=df.abstract_clean.str.lower()
T=df.title_clean.str.lower()
action=r'\b(?:we\s+)?(?:present|describe|develop(?:ed)?|design(?:ed)?|construct(?:ed)?|build|built|fabricat(?:e|ed)|introduce|demonstrate)\b'
object_rx=r'\b(?:instrument|system|setup|apparatus|device|platform|sensor|detector|source|generator|furnace|chamber|microscope|spectrometer|diagnostic|facility|station|probe|controller|pyrometer|reactor|oven|cryostat|cell|stage)\b'
dev=((A.str.contains(action,regex=True) & A.str.contains(object_rx,regex=True)) | 
     T.str.contains(r'\b(?:design|development|construction|instrumentation|apparatus|setup|system|platform|device|facility|prototype)\b',regex=True)) & ~df.notice
D=df[dev].copy()
print('Heuristic instrument-development subset:',len(D),f'({100*len(D)/len(research):.1f}% of research-like records)')

# Title dimension proxies. Categories overlap by design.
instrument_title=r'\b(?:instrument|system|setup|apparatus|device|platform|sensor|detector|source|generator|furnace|chamber|microscope|spectrometer|diagnostic|facility|station|probe|controller|pyrometer|reactor|oven|cryostat|cell|stage|camera|scanner|analyzer|goniometer|diffractometer|interferometer|magnetometer|calorimeter|thermometer|actuator|amplifier)\b'
technique_title=r'\b(?:method|technique|measurement|imaging|spectroscopy|microscopy|tomography|metrology|detection|diagnosis|diagnostic|monitoring|characterization|analysis|annealing|heating|cooling|calibration|control|feedback|modulation|sensing|spectrometry|diffracti(?:on|ometry)|interferometry|thermometry|pyrometry|electrochemistry|deposition|fabrication)\b'
# Application/purpose as explicit syntax: for X, to + verb, applied/application to, or "in [domain/context]".
application_title=r'\b(?:for|toward|towards)\b|\bto\s+(?:measure|detect|monitor|characterize|study|investigate|enable|achieve|improve|control|generate|produce|evaluate|determine|image|observe|test|calibrate|reconstruct|analyze|analyse|perform|induce)\b|\bapplications?\s+(?:to|in|for)\b|\bin\s+(?:high-energy|plasma|materials?|biology|biomedical|medical|quantum|fusion|accelerator|microfluidic|nuclear|surface|atmospheric|combustion|cryogenic)\b'
D['names_instrument']=D.title_clean.str.lower().str.contains(instrument_title,regex=True)
D['names_technique']=D.title_clean.str.lower().str.contains(technique_title,regex=True)
D['names_application']=D.title_clean.str.lower().str.contains(application_title,regex=True)
from statsmodels.stats.proportion import proportion_confint
for c in ['names_instrument','names_technique','names_application']:
    k=int(D[c].sum()); n=len(D); lo,hi=proportion_confint(k,n,alpha=.05,method='wilson')
    print(c,k,f'{100*k/n:.1f}% (Wilson 95% CI {100*lo:.1f}–{100*hi:.1f}%)')
for combo in [('names_instrument','names_technique'),('names_instrument','names_application'),('names_technique','names_application'),('names_instrument','names_technique','names_application')]:
    k=int(D[list(combo)].all(axis=1).sum()); print(' + '.join(combo),k,f'({100*k/len(D):.1f}%)')

print('\nRandom reproducible audit sample (seed 202603; 15):')
print(D.sample(15,random_state=202603)[['title_clean','doi','names_instrument','names_technique','names_application']].to_string(index=False))

Heuristic instrument-development subset: 2968 (69.3% of research-like records)
names_instrument 1711 57.6% (Wilson 95% CI 55.9–59.4%)
names_technique 1384 46.6% (Wilson 95% CI 44.8–48.4%)
names_application 1642 55.3% (Wilson 95% CI 53.5–57.1%)
names_instrument + names_technique 791 (26.7%)
names_instrument + names_application 1018 (34.3%)
names_technique + names_application 802 (27.0%)
names_instrument + names_technique + names_application 508 (17.1%)

Random reproducible audit sample (seed 202603; 15):
                                                                                                                       title_clean               doi  names_instrument  names_technique  names_application
                         Multimode objective lens for momentum microscopy and x-ray photoemission electron microscopy: Experiments 10.1063/5.0311293             False             True               True
                                   Multi-dimensional incoherent Thomson scattering sy

In [4]:
# Find domain and build/retrofit comparators by targeted corpus queries.
queries={
'induction':r'\binduction\b|radio[- ]frequency heating|\bRF heating\b',
'furnace_anneal':r'\bfurnace\b|\banneal(?:ing|ed)?\b|grain growth',
'vacuum_heat':r'\bvacuum\b.*\b(?:heat|heating|anneal|furnace|oven|temperature)\b|\b(?:heat|heating|anneal|furnace|oven|temperature)\b.*\bvacuum\b',
'open_lowcost':r'open[- ]source|low[- ]cost|cost[- ]effective|inexpensive|affordable',
'retrofit_modern':r'\bretrofit(?:ting|ted)?\b|\bmoderni[sz](?:e|ed|ation|ing)\b|\badapt(?:ing|ed|ation)\b|\bupgrade(?:d|s|ing)?\b|\bmodif(?:y|ied|ication)\b.*\bcommercial\b|\bcommercial\b.*\bmodif',
'labview_daq':r'\blabview\b|data acquisition|\bDAQ\b',
'pyrometer':r'pyromet(?:er|ry)|dual[- ]wavelength|two[- ]color'
}
text=(df.title_clean+' '+df.abstract_clean)
for name,rx in queries.items():
    hit=df[text.str.contains(rx,case=False,regex=True) & ~df.notice]
    print(f'\n### {name}: n={len(hit)}')
    print(hit[['year','title_clean','doi']].to_string(index=False,max_rows=80))



### induction: n=36
 year                                                                                                                                                       title_clean               doi
 2021                                                                                             A pulse oxidation facility for the study of oxide nucleation behavior 10.1063/5.0048536
 2021                             Rapid diagnosis and continuous monitoring of intracerebral hemorrhage with magnetic induction tomography based on stacked autoencoder 10.1063/5.0050171
 2021                                                      Simulation based study of magnetic velocity induction system by using Analysis System Electromagnetics Suite 10.1063/5.0050383
 2021                                                                                           Identification of object composition with magnetic inductive tomography 10.1063/5.0054263
 2021 Design, experiment, and performance analysi


### furnace_anneal: n=42
 year                                                                                                                                                                        title_clean               doi
 2021                                                                                          An actively compensated 8 nT-level magnetic shielding system for 10-m atom interferometer 10.1063/5.0053971
 2021                                                              A new high-pressure technique for the measurement of low frequency seismic attenuation using cyclic torsional loading 10.1063/5.0055549
 2021                                        Optimization on thermoelectric characteristics of indium tin oxide/indium oxide thin film thermocouples based on screen printing technology 10.1063/5.0057148
 2021                                                                     Image reconstruction method for electrical capacitance tomography using adaptive simulat


### vacuum_heat: n=129
 year                                                                                                                                                title_clean               doi
 2021                          Field-angle-dependent multi-frequency electron spin resonance spectroscopy in submillimeter wave range based on thermal detection 10.1063/5.0053227
 2021                                     A rapid compression machine coupled with time-resolved molecular beam mass spectrometry for gas-phase kinetics studies 10.1063/5.0055585
 2021                                                                                VIZSLA—Versatile Ice Zigzag Sublimation Setup for Laboratory Astrochemistry 10.1063/5.0061762
 2021                                                                                   Containerless metal single-crystal growth via electromagnetic levitation 10.1063/5.0064486
 2021                                                                            


### open_lowcost: n=194
 year                                                                                                                                                                title_clean               doi
 2021                                                        Single-pixel imaging: An overview of different methods to be used for 3D space reconstruction in harsh environments 10.1063/5.0050358
 2021                                                           Impact and trends in embedding field programmable gate arrays and microcontrollers in scientific instrumentation 10.1063/5.0050999
 2021                                                                                                 Partial discharge detection with on-chip spiral inductor as a loop antenna 10.1063/5.0051419
 2021                                                                         Fabrication of spectroscopic characterization techniques using an optical fiber-based spectrometer 10.1063/5.0054308



### labview_daq: n=132
 year                                                                                                                                                                                                      title_clean               doi
 2021                                                                                                     X-ray multi-probe data acquisition: A novel technique for laser pump x-ray transient absorption spectroscopy 10.1063/5.0050713
 2021                                                                                           A rapid compression machine coupled with time-resolved molecular beam mass spectrometry for gas-phase kinetics studies 10.1063/5.0055585
 2021                                                                                                             Development and assessment of an x-ray tube-based multi-beam x-ray scatter projection imaging system 10.1063/5.0055900
 2021                                       

In [5]:
# Concise targeted title searches and TF-IDF nearest neighbors.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

patterns={
'heating/furnace/annealing/grain growth':r'\b(induction[- ]heating|furnace|anneal(?:ing)?|grain growth|high[- ]temperature heating)\b',
'open/low-cost':r'\b(open[- ]source|low[- ]cost|cost[- ]effective|inexpensive|affordable)\b',
'retrofit/adapt/modernize/upgrade':r'\b(retrofit\w*|adapt(?:ing|ed|ation)|moderniz\w*|modernis\w*|upgrad\w*)\b',
'commercial equipment':r'\bcommercial\b',
}
for label,pat in patterns.items():
    hit=research[research.title_clean.str.contains(pat,case=False,regex=True)]
    print(f'\n{label}: {len(hit)} title hits')
    print(hit[['title_clean','doi']].to_string(index=False,max_rows=40))

# Similarity using title plus abstract, emphasizing proposed technical concepts.
query = proposed + ' high-temperature vacuum annealing furnace induction heating RF generator pyrometer temperature feedback graphite susceptor grain growth nickel open design commercial equipment retrofit'
corpus=(research.title_clean+' '+research.abstract_clean).tolist()
vec=TfidfVectorizer(stop_words='english',ngram_range=(1,2),min_df=2,max_df=.85,sublinear_tf=True)
X=vec.fit_transform(corpus+[query])
sims=cosine_similarity(X[-1],X[:-1]).ravel()
idx=np.argsort(sims)[::-1][:30]
print('\nTop 30 TF-IDF comparators:')
out=research.iloc[idx][['year','title_clean','doi']].copy(); out['similarity']=sims[idx]
print(out.to_string(index=False))

/tmp/ipykernel_20/2696466011.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit=research[research.title_clean.str.contains(pat,case=False,regex=True)]
/tmp/ipykernel_20/2696466011.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit=research[research.title_clean.str.contains(pat,case=False,regex=True)]
/tmp/ipykernel_20/2696466011.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit=research[research.title_clean.str.contains(pat,case=False,regex=True)]



heating/furnace/annealing/grain growth: 9 title hits
                                                                                                                                                                       title_clean               doi
                                                                    Image reconstruction method for electrical capacitance tomography using adaptive simulated annealing algorithm 10.1063/5.0059296
                                                                            Liver tumor ablation enhancement by induction-heating system with bitter-like deep magnetic field coil 10.1063/5.0066308
                                                                             Solidification furnace for in situ observation of bulk transparent systems and image analysis methods 10.1063/5.0150391
                                                                                    X-radiography front tracking gradient furnace for directional solidificati


Top 30 TF-IDF comparators:
 year                                                                                                                                                   title_clean               doi  similarity
 2024                                       Design, development, and performance of a versatile graphene epitaxy system for the growth of epitaxial graphene on SiC 10.1063/5.0194852    0.162183
 2022                                                        Liver tumor ablation enhancement by induction-heating system with bitter-like deep magnetic field coil 10.1063/5.0066308    0.136533
 2023                                                                                         A laser-based system to heat nuclear fuel pellets at high temperature 10.1063/5.0139508    0.134215
 2026                                                              Pressure-controlled vacuum chamber for photothermal processing of thin films via photonic curing 10.1063/5.0300385    0.114845
 2

In [6]:
# Quantify proposed and candidate discoverability terms as document frequencies in titles and abstracts.
term_patterns={
'RF':r'\bRF\b|\bradio[- ]frequency\b',
'radio-frequency (spelled out)':r'\bradio[- ]frequency\b',
'induction':r'\binduction\b',
'induction heating':r'\binduction[- ]heating\b|\binduction heating\b',
'generator':r'\bgenerator(?:s)?\b',
'vacuum':r'\bvacuum\b',
'vacuum annealing':r'\bvacuum anneal(?:ing)?\b',
'annealing':r'\banneal(?:ing|ed)?\b',
'annealing system':r'\bannealing system\b',
'grain growth':r'\bgrain growth\b',
'reactive metal(s)':r'\breactive[- ]metals?\b',
'computer-controlled':r'\bcomputer[- ]controlled\b',
'vacuum-integrated':r'\bvacuum[- ]integrated\b',
'retrofit*':r'\bretrofit\w*\b',
'modernization/modernisation':r'\bmoderni[sz]\w*\b',
'high-temperature':r'\bhigh[- ]temperature\b',
'pyrometer/pyrometry':r'\bpyromet(?:er|ers|ry)\b',
'optical pyrometer/pyrometry':r'\boptical pyromet(?:er|ers|ry)\b',
'dual-wavelength':r'\bdual[- ]wavelength\b',
'two-color/two-colour':r'\btwo[- ]colou?r\b',
'temperature feedback':r'\btemperature feedback\b',
'feedback control':r'\bfeedback control\b',
'graphite crucible':r'\bgraphite crucible\b',
'susceptor':r'\bsusceptor(?:s)?\b',
'open-source/open source':r'\bopen[- ]source\b',
'low-cost/low cost':r'\blow[- ]cost\b',
'design files':r'\bdesign files?\b',
'LabVIEW':r'\bLabVIEW\b',
'data acquisition/DAQ':r'\bdata acquisition\b|\bDAQ\b',
'nickel':r'\bnickel\b',
'ceramic(s)':r'\bceramics?\b',
'YSZ/yttria-stabilized zirconia':r'\bYSZ\b|yttria[- ]stabilized zirconia',
}
records=[]
for term,pat in term_patterns.items():
    records.append([term,int(research.title_clean.str.contains(pat,case=False,regex=True).sum()),
                    int(research.abstract_clean.str.contains(pat,case=False,regex=True).sum())])
term_df=pd.DataFrame(records,columns=['term','title_n','abstract_n'])
term_df['title_pct']=100*term_df.title_n/len(research)
term_df['abstract_pct']=100*term_df.abstract_n/research.abstract_clean.ne('').sum()
print(term_df.to_string(index=False,formatters={'title_pct':'{:.2f}'.format,'abstract_pct':'{:.2f}'.format}))

# Most common meaningful unigrams/bigrams in all titles, to contextualize search vocabulary.
v=TfidfVectorizer(stop_words='english',ngram_range=(1,2),binary=True,use_idf=False,norm=None,token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z-]+\b')
M=v.fit_transform(research.title_clean.str.lower())
freq=np.asarray(M.sum(axis=0)).ravel(); names=np.array(v.get_feature_names_out())
uni=np.array([len(x.split())==1 for x in names]); bi=~uni
print('\nTop title unigrams:',list(zip(names[uni][np.argsort(freq[uni])[-40:][::-1]],freq[uni][np.argsort(freq[uni])[-40:][::-1]].astype(int))))
print('\nTop title bigrams:',list(zip(names[bi][np.argsort(freq[bi])[-40:][::-1]],freq[bi][np.argsort(freq[bi])[-40:][::-1]].astype(int))))

                          term  title_n  abstract_n title_pct abstract_pct
                            RF       75         204      1.75         4.78
 radio-frequency (spelled out)       49         138      1.14         3.23
                     induction        8          32      0.19         0.75
             induction heating        1           1      0.02         0.02
                     generator       76         142      1.77         3.33
                        vacuum       67         321      1.56         7.52
              vacuum annealing        0           0      0.00         0.00
                     annealing        4          23      0.09         0.54
              annealing system        0           1      0.00         0.02
                  grain growth        0           2      0.00         0.05
             reactive metal(s)        0           1      0.00         0.02
           computer-controlled        1           9      0.02         0.21
             vacuum-integ


Top title unigrams: [('based', np.int64(522)), ('using', np.int64(418)), ('design', np.int64(366)), ('x-ray', np.int64(352)), ('measurement', np.int64(345)), ('method', np.int64(301)), ('high', np.int64(285)), ('imaging', np.int64(261)), ('measurements', np.int64(256)), ('development', np.int64(252)), ('spectroscopy', np.int64(251)), ('magnetic', np.int64(249)), ('laser', np.int64(238)), ('optical', np.int64(196)), ('ion', np.int64(183)), ('electron', np.int64(183)), ('temperature', np.int64(181)), ('analysis', np.int64(180)), ('plasma', np.int64(174)), ('beam', np.int64(173)), ('spectrometer', np.int64(170)), ('source', np.int64(167)), ('detection', np.int64(157)), ('field', np.int64(151)), ('experiments', np.int64(146)), ('neutron', np.int64(144)), ('control', np.int64(129)), ('characterization', np.int64(128)), ('frequency', np.int64(125)), ('situ', np.int64(123)), ('novel', np.int64(122)), ('scattering', np.int64(121)), ('study', np.int64(120)), ('compact', np.int64(119)), ('energ

In [7]:
# Inspect exact records most relevant to retrofit, heating, vacuum, temperature control, and open builds.
sel_dois=[
'10.1063/5.0066308','10.1063/5.0139508','10.1063/5.0202933','10.1063/5.0283346',
'10.1063/5.0290862','10.1063/5.0299443','10.1063/5.0272551','10.1063/5.0186155',
'10.1063/5.0141788','10.1063/5.0058786','10.1063/5.0191946','10.1063/5.0047652',
'10.1063/5.0050860','10.1063/5.0187818','10.1063/5.0119009','10.1063/5.0099778'
]
S=df[df.doi.isin(sel_dois)].copy()
print(S[['year','title_clean','doi']].sort_values(['year','doi']).to_string(index=False))
print('\nRetrofit/adaptation/modernization occurrences in title or abstract:')
rx=r'\b(?:retrofit\w*|adapt(?:ing|ed|ation)|moderni[sz]\w*|upgrad\w*)\b'
for _,r in research[(research.title_clean+' '+research.abstract_clean).str.contains(rx,case=False,regex=True)].iterrows():
    matches=[]
    for m in re.finditer(rx,r.title_clean+' '+r.abstract_clean,flags=re.I):
        matches.append((r.title_clean+' '+r.abstract_clean)[max(0,m.start()-65):m.end()+90])
    print('\n',r.title_clean,'|',r.doi,'\n ', ' ... '.join(matches[:3]))

# Structural common starts and connectors.
first=research.title_clean.map(lambda s:' '.join(words(s)[:2]).lower())
print('\nMost common two-token starts:')
print(first.value_counts().head(25).to_string())
for marker in [' based on ',' using ',' for ',' with ',' toward ',' via ',' of ',' in ']:
    n=research.title_clean.str.lower().str.contains(marker,regex=False).sum()
    print(repr(marker.strip()),n,f'({100*n/len(research):.1f}%)')

 year                                                                                                                                                                title_clean               doi
 2021                                                                                                          Scanning SQUID microscopy in a cryogen-free dilution refrigerator 10.1063/5.0047652
 2021                                                              Adapting a continuous flow cryostat and a plate DAC to do high pressure Raman experiments at low temperatures 10.1063/5.0050860
 2022 LithPulser: An open-source pulse generator with 1 ns time resolution based on the Red Pitaya STEMlab 125-10 featuring real-time conditional logic for experimental control 10.1063/5.0058786
 2022                                                                     Liver tumor ablation enhancement by induction-heating system with bitter-like deep magnetic field coil 10.1063/5.0066308
 2022                    


Most common two-token starts:
title_clean
development of         187
design and             124
design of               79
a novel                 62
research on             55
a new                   43
a compact               38
in situ                 28
measurement of          27
a versatile             27
development and         27
characterization of     26
calibration of          19
a high                  16
application of          15
evaluation of           15
implementation of       15
optimization of         15
a simple                14
investigation of        11
measurements of         11
analysis of             11
study on                10
study of                10
an improved             10
'based on' 451 (10.5%)
'using' 411 (9.6%)
'for' 2069 (48.3%)
'with' 668 (15.6%)
'toward' 18 (0.4%)
'via' 54 (1.3%)
'of' 2179 (50.9%)
'in' 889 (20.7%)


In [8]:
# Refine development-paper definition and quantify sensitivity to classification rule.
# Strict rule: explicit development/design term in title OR an abstract sentence where an author action
# (present/describe/develop/design/build...) occurs within 100 characters of an instrument-class noun.
def explicit_dev(text):
    sents=re.split(r'(?<=[.!?])\s+', text.lower())
    act=r'\b(?:we\s+)?(?:present|describe|develop(?:ed)?|design(?:ed)?|construct(?:ed)?|built|fabricat(?:e|ed)|introduce|report)\b'
    obj=r'\b(?:instrument|system|setup|apparatus|device|platform|sensor|detector|source|generator|furnace|chamber|microscope|spectrometer|diagnostic|facility|station|probe|controller|pyrometer|reactor|oven|cryostat|cell|stage)\b'
    return any(re.search(act+r'.{0,100}'+obj, s) or re.search(obj+r'.{0,100}'+act,s) for s in sents)

title_dev=r'\b(?:design|development|construction|instrumentation|apparatus|setup|system|platform|device|facility|prototype)\b'
strict=(research.title_clean.str.contains(title_dev,case=False,regex=True) | research.abstract_clean.map(explicit_dev))
Ds=research[strict].copy()
for x in [Ds,D]:
    x['names_instrument']=x.title_clean.str.lower().str.contains(instrument_title,regex=True)
    x['names_technique']=x.title_clean.str.lower().str.contains(technique_title,regex=True)
    x['names_application']=x.title_clean.str.lower().str.contains(application_title,regex=True)
print('Strict subset n=',len(Ds),f'({100*len(Ds)/len(research):.1f}%)','; broad n=',len(D))
for label,x in [('strict',Ds),('broad',D)]:
    vals=[]
    for c in ['names_instrument','names_technique','names_application']:
        k=int(x[c].sum()); lo,hi=proportion_confint(k,len(x),method='wilson')
        vals.append(f'{c.replace("names_","")} {k}/{len(x)}={100*k/len(x):.1f}% [{100*lo:.1f},{100*hi:.1f}]')
    all3=x[['names_instrument','names_technique','names_application']].all(axis=1).sum()
    print(label, '; '.join(vals), f'; all3={all3}/{len(x)}={100*all3/len(x):.1f}%')

# Targeted RF/generator and closest build-title records only.
for label,pat in {
'RF plus generator/heating/system':r'(?=.*\b(?:RF|radio[- ]frequency)\b)(?=.*\b(?:generator|heating|system|source|power)\b)',
'commercial in title':r'\bcommercial\b',
'retrofit in title':r'\bretrofit\w*\b',
'computer controlled in title':r'\bcomputer[- ]controlled\b',
}.items():
    h=research[research.title_clean.str.contains(pat,case=False,regex=True)]
    print('\n',label,len(h))
    print(h[['title_clean','doi']].to_string(index=False,max_rows=100))

# Candidate lengths.
candidates=[
'Computer-controlled radio-frequency induction furnace for high-temperature vacuum annealing',
'A computer-controlled radio-frequency induction furnace for high-temperature vacuum annealing and grain growth',
'Retrofitting a commercial radio-frequency induction generator for computer-controlled vacuum annealing',
'Open modernization of a commercial radio-frequency induction generator for vacuum annealing',
'A vacuum-integrated radio-frequency induction heating system with pyrometer feedback for grain growth',
]
print('\nCandidate lengths/percentiles:')
for s in candidates:
    w,c=len(words(s)),len(s)
    print(w,c,round(percentileofscore(research.n_words,w,kind='mean'),1),round(percentileofscore(research.n_chars,c,kind='mean'),1),s)

Strict subset n= 2254 (52.6%) ; broad n= 2968
strict instrument 1527/2254=67.7% [65.8,69.6]; technique 1044/2254=46.3% [44.3,48.4]; application 1292/2254=57.3% [55.3,59.3] ; all3=463/2254=20.5%
broad instrument 1711/2968=57.6% [55.9,59.4]; technique 1384/2968=46.6% [44.8,48.4]; application 1642/2968=55.3% [53.5,57.1] ; all3=508/2968=17.1%



 RF plus generator/heating/system 33
                                                                                                                                       title_clean               doi
                                     Breakthrough instruments and products RF/Microwave power amplifiers for electromagnetic compatibility testing 10.1063/5.0061251
                        Overview of diagnostics on a small-scale RF source for fusion (ROBIN) and the one planned for the diagnostic beam for ITER 10.1063/5.0076009
                                    Diagnostics of RF coupling in H− ion sources as a tool for optimizing source design and operational parameters 10.1063/5.0077934
               Convergent neutral gas injection using supersonic gas puffing (SSGP) method for propellant feeding system in RF electric propulsion 10.1063/5.0082821
                      Optimization of transverse emittance for RF-driven negative hydrogen ion source developed at China Spallation Neutr

In [9]:
# Final quantitative summary for exact core terms, title structures, and selected comparator abstracts.
core={
'furnace':r'\bfurnaces?\b','heating system':r'\bheating system\b','temperature control':r'\btemperature control\b',
'pyrometer/pyrometry':r'\bpyromet(?:er|ers|ry)\b','grain growth':r'\bgrain growth\b','annealing':r'\banneal(?:ing|ed)?\b',
'vacuum':r'\bvacuum\b','high-temperature':r'\bhigh[- ]temperature\b','radio frequency/RF':r'\bradio[- ]frequency\b|\bRF\b',
'induction heating':r'\binduction[- ]heating\b|\binduction heating\b','open-source':r'\bopen[- ]source\b','low-cost':r'\blow[- ]cost\b'
}
print('Core term document frequencies (research-like n=4,285; nonempty abstracts among these n=%d)'%research.abstract_clean.ne('').sum())
for k,p in core.items():
    print(f'{k:24s} title {research.title_clean.str.contains(p,case=False,regex=True).sum():3d}; abstract {research.abstract_clean.str.contains(p,case=False,regex=True).sum():3d}')

# Sentence-like conservative proxy: question, finite-verb-led title, or title containing an overt subject + finite verb pattern.
finite_lead=r'^(?:we|this|these|a|an|the)\s+\w+\s+(?:is|are|was|were|has|have|shows?|enables?|provides?|improves?|achieves?|demonstrates?|allows?|reveals?)\b'
finite_any=r'\b(?:we|this (?:method|system|instrument|device|approach|technique|study)|these (?:results|measurements))\s+(?:is|are|was|were|has|have|shows?|enables?|provides?|improves?|achieves?|demonstrates?|allows?|reveals?)\b'
sentence_like=research.title_clean.str.endswith('?')|research.title_clean.str.lower().str.contains(finite_lead+'|'+finite_any,regex=True)
print('\nConservative sentence-like titles:',sentence_like.sum(),f'({100*sentence_like.mean():.2f}%); nominal/non-sentence by proxy:',(~sentence_like).sum(),f'({100*(~sentence_like).mean():.2f}%)')
print(research.loc[sentence_like,['title_clean','doi']].head(20).to_string(index=False))

# Output selected comparators with title length and short abstract snippets around key concepts.
sel=['10.1063/5.0050860','10.1063/5.0113493','10.1063/5.0066308','10.1063/5.0202933','10.1063/5.0283346','10.1063/5.0290862','10.1063/5.0299443','10.1063/5.0141788','10.1063/5.0191946']
print('\nSelected comparators:')
for _,r in df[df.doi.isin(sel)].sort_values(['year','doi']).iterrows():
    print(f"{r.year} | {len(words(r.title_clean))}w/{len(r.title_clean)}c | {r.title_clean} | https://doi.org/{r.doi}")
    print('  ',r.abstract_clean[:320].replace('\n',' ')+'...')

Core term document frequencies (research-like n=4,285; nonempty abstracts among these n=4268)
furnace                  title   4; abstract  21
heating system           title   4; abstract   9


temperature control      title   8; abstract  39
pyrometer/pyrometry      title   5; abstract  14
grain growth             title   0; abstract   2


annealing                title   4; abstract  23
vacuum                   title  67; abstract 321
high-temperature         title  52; abstract 135


radio frequency/RF       title  75; abstract 204
induction heating        title   1; abstract   1
open-source              title   8; abstract  25


low-cost                 title   4; abstract  85

Conservative sentence-like titles: 2 (0.05%); nominal/non-sentence by proxy: 4283 (99.95%)
                                                                                                title_clean               doi
                                                 How does crosstalk influence the jitter of a clock signal? 10.1063/5.0191289
How does the limited resolution of space plasma analyzers affect the accuracy of space plasma measurements? 10.1063/5.0218667

Selected comparators:
2021 | 18w/109c | Adapting a continuous flow cryostat and a plate DAC to do high pressure Raman experiments at low temperatures | https://doi.org/10.1063/5.0050860
   We present a method for modifying a continuous flow cryostat and a steel plate DAC (Diamond Anvil Cell) to perform high pressure micro-Raman experiments at low temperatures. Despite using a steel DAC with a lower specific heat capacity (∼335 J/kg K), this setup can routinely perform high pr